# 🔍 Notebook 2: Hybrid RAG Pipeline
## Dense Retrieval + BM25 Sparse Retrieval + Reciprocal Rank Fusion

This notebook demonstrates our **hybrid retrieval system** that combines three signals:

| Signal | Method | Catches |
|---|---|---|
| **Dense** | ChromaDB cosine similarity on MiniLM embeddings | Semantic meaning |
| **Sparse** | BM25 keyword matching | Exact terms |
| **Fusion** | Reciprocal Rank Fusion (RRF) | Best of both worlds |

**RRF Formula:** `RRF_score(doc) = Σ 1/(k + rank_dense) + 1/(k + rank_sparse)` where k=60

In [ ]:
import sys
sys.path.insert(0, '..')

from src.rag.embedder import Embedder
from src.rag.vector_store import HybridVectorStore, reciprocal_rank_fusion
import numpy as np

print('✅ RAG pipeline modules loaded!')

## Step 1: Embedding Generation

We use `all-MiniLM-L6-v2` from SentenceTransformers (384 dimensions, ~90MB):
- **Free** — no API key needed
- **Fast** — runs on CPU
- **Effective** — trained on 1B+ sentence pairs

In [ ]:
# Initialize the embedder
embedder = Embedder()

# Demo: Embed sample texts and compute similarity
test_texts = [
    'Machine learning is a subset of artificial intelligence.',
    'Deep learning uses neural networks with many layers.',
    'Cats are popular pets around the world.',
    'The transformer architecture revolutionized NLP.',
    'Python is a programming language for data science.',
]

embeddings = embedder.embed_batch(test_texts)
print(f'Embedding dimension: {len(embeddings[0])}')
print(f'Number of embeddings: {len(embeddings)}')

# Compute similarity matrix
from numpy.linalg import norm
def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

print(f'\n--- Most Similar Pairs ---')
for i, t1 in enumerate(test_texts):
    sims = [cosine_sim(np.array(embeddings[i]), np.array(embeddings[j])) for j in range(len(test_texts))]
    top = sorted(range(len(sims)), key=lambda x: sims[x], reverse=True)[1]
    print(f"  '{t1[:45]}...' → '{test_texts[top][:45]}...' (sim={sims[top]:.3f})")

## Step 2: Reciprocal Rank Fusion Demo

RRF is **rank-based, not score-based**. This matters because dense similarity scores (0.0–1.0) and BM25 scores (unbounded) are on completely different scales. RRF normalizes both to ranks.

In [ ]:
# Demonstrate RRF fusion with a toy example
dense_results = [
    {'id': 'doc_A', 'text': 'Neural networks learn representations', 'score': 0.92},
    {'id': 'doc_B', 'text': 'Machine learning algorithms', 'score': 0.85},
    {'id': 'doc_C', 'text': 'Backpropagation trains networks', 'score': 0.78},
]

sparse_results = [
    {'id': 'doc_B', 'text': 'Machine learning algorithms', 'score': 12.5},
    {'id': 'doc_D', 'text': 'BERT transformer model by Google', 'score': 8.3},
    {'id': 'doc_A', 'text': 'Neural networks learn representations', 'score': 5.1},
]

fused = reciprocal_rank_fusion(dense_results, sparse_results, k=60)

print('=== Dense Ranking ===')
for i, r in enumerate(dense_results):
    print(f"  Rank {i+1}: {r['id']} (score={r['score']:.2f}) - {r['text']}")

print('\n=== BM25 (Sparse) Ranking ===')
for i, r in enumerate(sparse_results):
    print(f"  Rank {i+1}: {r['id']} (score={r['score']:.2f}) - {r['text']}")

print('\n=== RRF Fused Ranking ===')
for i, r in enumerate(fused):
    print(f"  Rank {i+1}: {r['id']} (RRF={r.get('rrf_score', 0):.4f}) - {r['text']}")

print('\n💡 doc_B appears in BOTH lists → RRF boosts it to the top!')
print('   doc_D only has a keyword match but still makes the fused list.')

## Step 3: End-to-End RAG Query

The full pipeline: **query → embed → hybrid retrieve → context assembly → LLM generation**

> **Note:** This step requires ingested data and an LLM provider configured in `.env`.

In [ ]:
# Retrieval-only demo (no LLM required)
try:
    from src.rag.pipeline import retrieve
    results = retrieve('What is retrieval augmented generation?')
    print(f'Retrieved {len(results)} chunks:\n')
    for i, chunk in enumerate(results[:5]):
        text = chunk.get('text', chunk.get('content', str(chunk)))[:150]
        source = chunk.get('metadata', {}).get('source', 'unknown')
        print(f'  [{i+1}] (source: {source})')
        print(f'      {text}...\n')
except Exception as e:
    print(f'⚠️ Retrieval requires ingested data.')
    print(f"   Run 'python scripts/ingest.py --file urls.txt' first.")
    print(f'   Error: {e}')